In [1]:
from dotenv import load_dotenv
import os

dotenv_path = os.path.join(os.getcwd(), '..', '.env')
load_dotenv(dotenv_path=dotenv_path)

API_KEY = os.getenv("GEMINI_API_KEY")
print(f"Key load: {'YES ✅' if API_KEY else 'NO ❌'}")

Key load: YES ✅


In [2]:
# importing the prompt from src folder we created to save the prompts
import sys
sys.path.append(os.path.join(os.getcwd(), '..'))
from src.prompts import easyf1_strategy_prompt_V2

print ("prompt loaded successfully")
print(f"prompt_length:{len(easyf1_strategy_prompt_V2)} characters")

prompt loaded successfully
prompt_length:1107 characters


In [8]:
import requests

def chat_easyf1(conversation_history, user_message):
    """Sends message to Gemini and maintains a  full conversation memory
     How memory works: 
      Everytime, you send a message. we inlcude all previous messages too. Gemini actually does not 
        rememeber anything, we are jusy handing it the full conversation transcript everytime 
           like giving someone a copy of all previous emails before they reply
    parameteres: 
    conversation_history: list 
    A list of all previous messages in the conversation.
    Starts empty[], grows with every exchange.
    
    user_message: str
    The new question or scenario from the user 
    
    Returns:
    
    reply: str
    Gemini's response text.
    
    conversation_history: list
    The updated history including this new exchange.
    Always pass this back on the next call.
        """
    # Step 1: Adding the new user message to history
    # Appending a dictionary with two keys:
    # "role" "tells Gemini who is speaking (user or model)
    # "parts" contains the actual message text 
    conversation_history.append({
        'role': 'user',
        'parts': [{"text": user_message}]
    })

    # Step 2: Building the URL with API key 
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-lite:generateContent?key={API_KEY}"

    # Step 3: Build the payload (Letter we are sned ing ti the Gemini)
    # Notice "contents" now contain the full history, not just one msg
    # this is how memory works, every call gets the full transcript 

    payload = {
        "contents": conversation_history,
        "systemInstruction":{
            "parts": [{"text": easyf1_strategy_prompt_V2}]
        }

    }

    #Step 4: Send the request
    #Adding timeout=30, give up if no response in 30 seconds,this prevents the hanging issue 
    response = requests.post(url, json=payload, timeout=30)
    result = response.json()


    # Step 5: Check the errors
    if "error"in result:
        error_message = f"Error {result['error']['code']}: [result{'error'},{'message'},[:150]]"
        conversation_history.pop()
        return error_message, conversation_history
    
    # Step 6: Extract the reply text from the nested response
    reply = result['candidates'][0]['content']['parts'][0]['text']

    # Step 7: Add Gemini's reply to history too
    # Role is 'model' not 'assistant' for Gemini 
    #Next time we call this function, Genini will see its own previous reply and can build on it 

    conversation_history.append({
        "role": "model",
        "parts": [{"text": reply}]
    })

    #Step 8:Return both the reply and the updated history 
    #Note: "Always us the returned history on the next call"
    return reply, conversation_history

print ("Chat function is now ready")



Chat function is now ready


In [9]:
# Test the memory 
# Starting with an empty history --- a brand new conversation
history =[]
print ("Turn 1 -- setting the scene")

reply1, history = chat_easyf1(
    history,
    "It's lap 32 at silverstone. Hamilton is on 18 lap old medium tyres. safety car just deployed," \
    "verstappen pitted last lap for hards. what should mercedes do?"
)
print (reply1)

Turn 1 -- setting the scene
VERDICT: PIT HAMILTON
CONFIDENCE: HIGH
REASONING:
1.  **Tyre State:** Hamilton is on 18-lap old Mediums. While not critically degraded, they are past their optimal window, especially considering the tyre degradation expected at Silverstone.
2.  **Race Situation:** A Safety Car has been deployed. This significantly reduces the time loss during a pit stop, making it a prime opportunity for a "cheap" stop.
3.  **Pit Window Value:** Under the Safety Car, the time lost in the pit lane is drastically reduced from a typical ~22-25 seconds to potentially 10-12 seconds, depending on track position relative to the SC train.
4.  **Rivals:** Verstappen has *just* pitted for Hards. This means he has fresh tyres but potentially less ultimate pace than fresh Mediums or Softs if Hamilton were to pit for them. Crucially, Verstappen is now on the *hardest* compound, which may limit his ability to push hard for the entire remaining stint compared to someone on Mediums. Pitting

In [12]:
print ("Turn 2 --- Follow up question (no context repeated)")
# we are not repeating the race scenario 
# the model should remember it from Turn 1
reply2, history = chat_easyf1(
    history,
    "why did you choose that tyre compound specifically over soft "
)
print(reply2)

Turn 2 --- Follow up question (no context repeated)
That's an excellent and very pertinent follow-up question. It highlights the critical trade-off between outright pace and tyre longevity, especially at a circuit like Silverstone.

Here's a more detailed breakdown of why **Mediums** are generally the preferred choice over Softs in that specific scenario at Silverstone:

1.  **Silverstone's Tyre Degradation Profile:** Silverstone is notorious for its high-speed, sweeping corners (like Copse, Maggotts, Becketts, and Stowe) that exert immense lateral G-forces on the tyres. This leads to very rapid wear and thermal degradation, particularly for the softer compounds. A set of Softs on a car that's being pushed hard could easily start to lose significant performance after just 8-12 laps of racing, sometimes even sooner.

2.  **The Remaining Race Distance:** At lap 32, we're looking at potentially 18-28 laps of racing remaining after the Safety Car period.
    *   **Softs:** If Hamilton pitt

In [13]:
print("Turn 3 --- change the scenario")
#Tesitng if it can reason about a change to the same scenario
reply3, history = chat_easyf1(
    history,
    "what changes if it starts raining on lap 3"
) 
print(reply3)

Turn 3 --- change the scenario
Okay, let's analyze the impact of rain starting on Lap 3 at Silverstone, given the initial scenario: Hamilton on 18-lap old Mediums, Safety Car deployed, Verstappen pitted last lap for Hards.

This changes *everything*. Rain dramatically alters the tyre strategy landscape.

**Initial Assessment (with rain):**

1.  **Tyre State:** Hamilton's 18-lap old Mediums are now a massive liability in the wet. They are not designed for significant rainfall and will offer very little grip, making them dangerous and slow.
2.  **Race Situation:** Safety Car deployed. This is actually a *huge* benefit now, as it neutralizes the race and allows teams to react to the changing conditions without losing massive amounts of time.
3.  **Pit Window Value:** Under a Safety Car, the pit loss is minimal (as established before). In the wet, this becomes an *opportunity* to switch to the correct tyres.
4.  **Rivals:** Verstappen is on Hards. Hards are generally poor in the wet compar

In [14]:
# ---- SEE WHAT'S IN MEMORY ----
# This shows you the full conversation history as Gemini sees it.
# Each item in the list is one message — user or model.

print(f"Total messages in memory: {len(history)}")
print()

for i, message in enumerate(history):
    # Who is speaking?
    speaker = "YOU" if message["role"] == "user" else "GEMINI"
    # First 100 characters of the message
    preview = message["parts"][0]["text"][:100]
    print(f"Message {i+1} [{speaker}]: {preview}...")
    print()

Total messages in memory: 10

Message 1 [YOU]: It's lap 32 at silverstone. Hamilton is on 18 lap old medium tyres. safety car just deployed,verstap...

Message 2 [GEMINI]: VERDICT: PIT HAMILTON
CONFIDENCE: HIGH
REASONING:
1.  **Tyre State:** Hamilton is on 18-lap old Medi...

Message 3 [YOU]: why did you choose that tyre compound specifically over soft ...

Message 4 [GEMINI]: Ah, that's an excellent question, and it highlights the nuance of tyre strategy!

My recommendation ...

Message 5 [YOU]: why did you choose that tyre compound specifically over soft ...

Message 6 [GEMINI]: That's an excellent question, and it highlights the strategic thinking that goes into tyre choice, e...

Message 7 [YOU]: why did you choose that tyre compound specifically over soft ...

Message 8 [GEMINI]: That's an excellent and very pertinent follow-up question. It highlights the critical trade-off betw...

Message 9 [YOU]: what changes if it starts raining on lap 3...

Message 10 [GEMINI]: Okay, let's an